# Siemens Energy Job Scraper
Scrapes jobs.siemens-energy.com, scores against CV, writes to SQLite.

**Run order:** Execute cells top to bottom. Sections 3 and 5 make network/DB calls — don't re-run accidentally.

**To reuse for another company:** copy this notebook, update Section 2 only.

## 0. Imports & setup

In [1]:
import sys, os
import requests
from bs4 import BeautifulSoup
import re
import time
import json
import sqlite3
from datetime import date
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Make config_isis.py importable — assumes notebook is in repo root
sys.path.append(os.getcwd())
from config_isis import (
    CV_TEXT, HIGH_WEIGHT_TERMS, MEDIUM_WEIGHT_TERMS,
    IGNORE_TERMS, ALLOWED_LOCATIONS, MIN_SCORE, KEYWORD_BONUS_CAP
)

print('Imports OK')
print(f'Config loaded — {len(HIGH_WEIGHT_TERMS)} high-weight terms, {len(IGNORE_TERMS)} ignore terms')

Imports OK
Config loaded — 21 high-weight terms, 27 ignore terms


## 1. CV profile & scoring config
Loaded from `config_isis.py`. Nothing to edit here.

Run this cell to verify what was imported.

In [2]:
print('=== CV TEXT (first 300 chars) ===')
print(CV_TEXT[:300])
print()
print('=== HIGH WEIGHT TERMS ===')
print(HIGH_WEIGHT_TERMS)
print()
print('=== ALLOWED LOCATIONS ===')
print(ALLOWED_LOCATIONS)

=== CV TEXT (first 300 chars) ===

Senior Analytics und Insights Führungskraft mit 10 Jahren Erfahrung im E-Commerce.
SQL Experte BigQuery Datenanalyse Business Intelligence KPI Kennzahlen Reporting.
Kundenfeedback Kundeninsights Voice of Customer VoC NPS Retourenanalyse Funnel-Analyse.
Machine Learning ML Python scikit-learn NLP Te

=== HIGH WEIGHT TERMS ===
['sql', 'datenanalyse', 'data analysis', 'analytics', 'business intelligence', 'bi', 'looker', 'product analytics', 'kundeninsights', 'customer insights', 'kpi', 'dashboard', 'reporting', 'e-commerce', 'ecommerce', 'digital', 'retouren', 'merchandising', 'digitalisierung', 'daten', 'data']

=== ALLOWED LOCATIONS ===
['berlin', 'wien', 'vienna', 'remote', 'wo du willst', 'home office', 'homeoffice']


## 2. Scraper config (company-specific)
**This is the only section to rewrite when copying this notebook for a new company.**

In [ ]:
# ── Identity ──────────────────────────────────────────────────────────────────
SOURCE = 'siemens_energy'   # Written to the 'source' column in SQLite
DB_PATH = 'data/jobs.db'    # Path to SQLite DB, relative to repo root

# ── Search URL ────────────────────────────────────────────────────────────────
# Base search URL with location filters already applied (Berlin + Wien).
# 29454 = location filter, 29455 = experience level filter
# folderOffset is added dynamically during pagination
SEARCH_BASE_URL = (
    'https://jobs.siemens-energy.com/de_DE/jobs/Jobs/'
    '?29454=964485&29454_format=11381'
    '&29455=964702&29455_format=11382'
    '&listFilterMode=1'
    '&folderRecordsPerPage=20'
    '&folderOffset={}'
)

# ── Pagination ────────────────────────────────────────────────────────────────
JOBS_PER_PAGE = 20
TOTAL_JOBS = 94      # Update if this changes — check the search results page
import math
TOTAL_PAGES = math.ceil(TOTAL_JOBS / JOBS_PER_PAGE)

# ── Timing ────────────────────────────────────────────────────────────────────
SLEEP_BETWEEN_PAGES = 1.5
SLEEP_BETWEEN_DETAILS = 1.0

# ── Company-specific keyword overrides ───────────────────────────────────────
# These are MERGED with your global config — not replacements
EXTRA_IGNORE = [
    'schweißer', 'monteur', 'elektriker', 'mechaniker',
    'techniker', 'ingenieur', 'konstruktion','graduate program','praktikant','mechatroniker'
]  # Add Siemens Energy-specific noise titles here

EXTRA_HIGH_WEIGHT = []   # e.g. ['energiewende', 'dekarbonisierung'] if relevant
EXTRA_MEDIUM_WEIGHT = [] 

# Merge with global config
IGNORE_TERMS_FINAL = IGNORE_TERMS + EXTRA_IGNORE
HIGH_WEIGHT_FINAL = HIGH_WEIGHT_TERMS + EXTRA_HIGH_WEIGHT
MEDIUM_WEIGHT_FINAL = MEDIUM_WEIGHT_TERMS + EXTRA_MEDIUM_WEIGHT

print(f'Source: {SOURCE}')
print(f'Total jobs to scrape: {TOTAL_JOBS} across {TOTAL_PAGES} pages')
print(f'Ignore terms (total): {len(IGNORE_TERMS_FINAL)}')

Source: siemens_energy
Total jobs to scrape: 94 across 5 pages
Ignore terms (total): 35


## 3. Scrape listing pages
Fetches all search result pages. Extracts title, link, job_id, location.

⚠️ **Network calls happen here.** Don't re-run unless you want to re-scrape.

In [4]:
def fetch_listing_page(offset):
    """Fetch one search results page. Returns list of raw job dicts."""
    url = SEARCH_BASE_URL.format(offset)
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f'  ⚠️  Offset {offset} failed: {e}')
        return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    jobs = []

    for article in soup.select('article.article--result'):
        # Title and link
        a_tag = article.select_one('a.article__header__focusable')
        if not a_tag:
            continue
        title = a_tag.get_text(strip=True)
        link = a_tag.get('href', '')

        # Job ID from data-job-id attribute
        content_div = article.select_one('div.article__content')
        job_id = content_div.get('data-job-id', '') if content_div else ''

        # Location — not in listing HTML, extracted from detail page later
        # We set a placeholder here
        jobs.append({
            'source': SOURCE,
            'source_job_id': job_id,
            'title': title,
            'link': link,
            'location': '',       # filled in Section 5
            'company': 'Siemens Energy',
            'start_date': '',
            'description': '',    # filled in Section 5
        })

    return jobs


# ── Run scrape ────────────────────────────────────────────────────────────────
raw_jobs = []
print(f'Scraping {TOTAL_PAGES} pages...')

for page_num, offset in enumerate(range(0, TOTAL_JOBS, JOBS_PER_PAGE), 1):
    print(f'  Page {page_num}/{TOTAL_PAGES} (offset={offset})...', end=' ')
    jobs = fetch_listing_page(offset)
    raw_jobs.extend(jobs)
    print(f'{len(jobs)} jobs')
    time.sleep(SLEEP_BETWEEN_PAGES)

# Deduplicate by source_job_id
seen = set()
unique_jobs = []
for j in raw_jobs:
    if j['source_job_id'] not in seen:
        seen.add(j['source_job_id'])
        unique_jobs.append(j)

print(f'\n✅ {len(unique_jobs)} unique jobs scraped')

Scraping 5 pages...
  Page 1/5 (offset=0)... 20 jobs
  Page 2/5 (offset=20)... 20 jobs
  Page 3/5 (offset=40)... 20 jobs
  Page 4/5 (offset=60)... 20 jobs
  Page 5/5 (offset=80)... 16 jobs

✅ 95 unique jobs scraped


In [5]:
# Inspect raw results before continuing
df_raw = pd.DataFrame(unique_jobs)
print(f'Shape: {df_raw.shape}')
df_raw[['title', 'source_job_id', 'link']].head(10)

Shape: (95, 8)


,title,source_job_id,link
0,"Werkstudent (w/m/d) - Pipeline Management, Das...",294237,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
1,Service Spezialist (Marine Antriebssysteme) (w...,286506,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
2,Corporate Commodity Manager (f/m/d) – Steel,292599,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
3,Logistikcore Mitarbeiter (w/m/d),293933,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
4,Rohrleitungsmonteur (m/w/d) im weltweiten Auße...,294203,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
5,IPLM Lead (w/m/d) for Large GasTurbine,294424,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
6,Senior Data & Knowledge Specialist (m/f/d),293754,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
7,Werkstudent (w/m/d) Interne Cloud Tool Einführung,294341,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
8,Bid Manager (f/m/d) Long Term Program Contracts,293940,https://jobs.siemens-energy.com/de_DE/jobs/Fol...
9,AM Fertigungsmitarbeiter,293717,https://jobs.siemens-energy.com/de_DE/jobs/Fol...


## 4. Filter
Drops irrelevant jobs by title before fetching descriptions.

Check the 'dropped' output — if real jobs are being filtered, update EXTRA_IGNORE in Section 2.

In [6]:
def is_irrelevant(title):
    title_lower = title.lower()
    return any(term in title_lower for term in IGNORE_TERMS_FINAL)

before = len(unique_jobs)
filtered_jobs = [j for j in unique_jobs if not is_irrelevant(j['title'])]
dropped = [j for j in unique_jobs if is_irrelevant(j['title'])]

print(f'Before filter: {before}')
print(f'After filter:  {len(filtered_jobs)}')
print(f'Dropped:       {len(dropped)}')
print()
print('--- Dropped titles (sanity check) ---')
for j in dropped:
    print(f"  {j['title']}")

Before filter: 95
After filter:  72
Dropped:       23

--- Dropped titles (sanity check) ---
  Werkstudent (w/m/d) - Pipeline Management, Dashboard, digitale Lösungen Offer Management
  Rohrleitungsmonteur (m/w/d) im weltweiten Außendienst
  Werkstudent (w/m/d) Interne Cloud Tool Einführung
  Werkstudent (w/m/d) Fullstack Webentwicklung - Quality Inspection Technology
  Siemens Energy Graduate Program - Transformation Manager (m/f/d)
  Werkstudent (w/m/d) UI/UX Design
  Werkstudent (w/m/d) Erstellung technischer Dokumentationen
  Siemens Energy Graduate Program - Process Engineer in Manufacturing (m/f/d)
  Werkstudent (w/m/d) zur Unterstützung bei der Abwicklung von Langzeitwartungsverträgen
  Siemens Energy Graduate Program - Spare Parts Package Analyst (f/m/d)
  Werkstudent (w/m/d) im Bereich Einkaufsleitung / Procurement für Gasturbinen
  Elektriker / Inbetriebnehmer (w/m/d)
  Montageingenieur (w/m/d) Gasturbine Erstinstallation im weltweiten Außendienst
  Elektroingenieur Hochspann

## 5. Fetch descriptions
Fetches each job's detail page and extracts full JD text + location.

⚠️ **Slow cell — ~1 sec per job.** A checkpoint JSON is saved after this cell so you don't need to re-fetch if scoring fails.

In [14]:
import os

checkpoint_path = f'data/{SOURCE}_checkpoint.json'

# Ensure directory exists every time
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

with open(checkpoint_path, 'w', encoding='utf-8') as f:
    json.dump(enriched_jobs, f, ensure_ascii=False, indent=2)

In [15]:
def fetch_description(job):
    """
    Fetches job detail page. Extracts:
    - description: all text from article__content__view__field__value divs containing <p> tags
    - location: from posting-location fields
    Returns (description, location) tuple.
    """
    if not job.get('link'):
        return job['title'], ''

    try:
        resp = requests.get(job['link'], timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"    ⚠️  Failed: {job['title'][:40]}: {e}")
        return job['title'], ''

    soup = BeautifulSoup(resp.text, 'html.parser')

    # Description: field value divs that contain <p> tags (the actual JD content)
    desc_parts = []
    for div in soup.select('div.article__content__view__field__value'):
        if div.find('p'):  # only divs with paragraph content
            desc_parts.append(div.get_text(separator=' ', strip=True))
    description = ' '.join(desc_parts)
    description = re.sub(r'\s+', ' ', description).strip()

    # Location: posting-location divs
    location_parts = []
    for div in soup.select('div.posting-location div.article__content__view__field__value'):
        text = div.get_text(strip=True)
        if text:
            location_parts.append(text)
    location = ', '.join(dict.fromkeys(location_parts))  # deduplicate while preserving order

    return description, location


# ── Run enrichment ────────────────────────────────────────────────────────────
enriched_jobs = []
print(f'Fetching descriptions for {len(filtered_jobs)} jobs...')

for i, job in enumerate(filtered_jobs, 1):
    print(f'  [{i}/{len(filtered_jobs)}] {job["title"][:50]}...', end=' ')
    desc, location = fetch_description(job)
    job['description'] = desc
    job['location'] = location
    enriched_jobs.append(job)
    print(f'{len(desc.split())} words | {location}')
    time.sleep(SLEEP_BETWEEN_DETAILS)

# ── Checkpoint save ───────────────────────────────────────────────────────────
checkpoint_path = f'data/{SOURCE}_checkpoint.json'
with open(checkpoint_path, 'w', encoding='utf-8') as f:
    json.dump(enriched_jobs, f, ensure_ascii=False, indent=2)
print(f'\n✅ Descriptions fetched. Checkpoint saved to {checkpoint_path}')

Fetching descriptions for 72 jobs...
  [1/72] Service Spezialist (Marine Antriebssysteme) (w/m/d... 774 words | Deutschland, Berlin
  [2/72] Corporate Commodity Manager (f/m/d) – Steel... 692 words | Deutschland, Berlin
  [3/72] Logistikcore Mitarbeiter (w/m/d)... 529 words | Deutschland, Berlin
  [4/72] IPLM Lead (w/m/d) for Large GasTurbine... 729 words | Deutschland, Berlin
  [5/72] Senior Data & Knowledge Specialist (m/f/d)... 601 words | Deutschland, Berlin
  [6/72] Bid Manager (f/m/d) Long Term Program Contracts... 689 words | Deutschland, Bayern, Erlangen
  [7/72] AM Fertigungsmitarbeiter... 368 words | Deutschland, Berlin
  [8/72] Betriebliche Sozialberater / Mitarbeitendenberater... 696 words | Deutschland, Berlin
  [9/72] Global Time & Attendance Process Consultant (f/m/d... 762 words | Deutschland, Bayern, Erlangen
  [10/72] Materials Engineer (w/m/d)... 604 words | Deutschland, Berlin
  [11/72] Product Cost Controller (w/m/d) Blades and Vanes... 547 words | Deutschland, Ber

In [ ]:
# If you need to reload from checkpoint instead of re-scraping:
# with open(f'data/{SOURCE}_checkpoint.json', encoding='utf-8') as f:
#     enriched_jobs = json.load(f)
# print(f'Loaded {len(enriched_jobs)} jobs from checkpoint')

In [ ]:
"""
#temporary, since I forgot to add a "data folder" - import json
with open('data/siemens_energy_checkpoint.json', 'w', encoding='utf-8') as f:
    json.dump(enriched_jobs, f, ensure_ascii=False, indent=2)
print('Checkpoint saved.')
"""

Checkpoint saved.


## 6. Location filter
Now that we have real location data from detail pages, apply the filter.

Kept separate from Section 4 because location isn't available until descriptions are fetched.

In [16]:
def is_allowed_location(location):
    loc_lower = location.lower()
    return any(allowed in loc_lower for allowed in ALLOWED_LOCATIONS)

before = len(enriched_jobs)
location_filtered = [j for j in enriched_jobs if is_allowed_location(j['location'])]
dropped_location = [j for j in enriched_jobs if not is_allowed_location(j['location'])]

print(f'Before location filter: {before}')
print(f'After location filter:  {len(location_filtered)}')
print()
print('--- Dropped (wrong location) ---')
for j in dropped_location:
    print(f"  {j['title'][:50]} | {j['location']}")

Before location filter: 72
After location filter:  51

--- Dropped (wrong location) ---
  Bid Manager (f/m/d) Long Term Program Contracts | Deutschland, Bayern, Erlangen
  Global Time & Attendance Process Consultant (f/m/d | Deutschland, Bayern, Erlangen
  Data Governance Specialist (f/m/d) | Deutschland, Bayern, Erlangen
  Data Architect (f/m/d) | Deutschland, Bayern, Erlangen
  AI / GenAI Solutions Architect (AWS) (f/m/d) | Deutschland, Bayern, Erlangen
  Digital Solution Analyst (f/m/d) - SAP S4/HANA | Deutschland, Bayern, Erlangen
  Global Commodity Manager (w/m/d) Facility Manageme | Deutschland, Bayern, Erlangen
  Procurement Project Manager (w/m/d) Real Estate Co | Deutschland, Bayern, Erlangen
  Supplier Quality Engineer (f/m/d) | Deutschland, Nordrhein-Westfalen, Muelheim an der Ruhr
  Risk Manager (f/m/d) Procurement of Gas Turbines | Deutschland, Nordrhein-Westfalen, Muelheim an der Ruhr
  IT Service Manager (m/f/d) Corporate Master Data ( | Deutschland, Nordrhein-Westfalen,

## 7. Score
TF-IDF cosine similarity against your CV profile + keyword bonus.

In [17]:
def keyword_bonus(text):
    text_lower = text.lower()
    bonus = 0
    for term in HIGH_WEIGHT_FINAL:
        if term in text_lower:
            bonus += 8
    for term in MEDIUM_WEIGHT_FINAL:
        if term in text_lower:
            bonus += 3
    return min(bonus, KEYWORD_BONUS_CAP)


def score_jobs(jobs):
    if not jobs:
        print('No jobs to score.')
        return jobs

    descriptions = [j['description'] for j in jobs]
    corpus = [CV_TEXT] + descriptions

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(corpus)
    similarities = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1:])[0]

    for i, job in enumerate(jobs):
        base = round(float(similarities[i]) * 100, 1)
        bonus = keyword_bonus(job['description'])
        job['score'] = min(round(base + bonus, 1), 100)

    return sorted(jobs, key=lambda x: x['score'], reverse=True)


scored_jobs = score_jobs(location_filtered)
print(f'✅ Scored {len(scored_jobs)} jobs')

✅ Scored 51 jobs


In [18]:
# Inspect results
df_scored = pd.DataFrame(scored_jobs)[['score', 'title', 'location', 'link']]
df_scored.head(20)

,score,title,location,link
0,36.7,Product Safety Officer (f/m/d),"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
1,35.7,Pflichtpraktikant (w/m/d) - Leadership & Organ...,"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
2,35.2,IPLM Lead (w/m/d) for Large GasTurbine,"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
3,35.2,Head (f/m/d) of Applied Engineering & Testing,"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
4,34.4,Global Order Manager (f/m/d),"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
5,33.3,Commercial Project Manager (f/m/d) mit Digital...,"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
6,32.7,Global Factory Network Costing Professional (m...,"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
7,32.6,Logistikcore Mitarbeiter (w/m/d),"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
8,32.5,Projektleiter Verfahrenstechnik in der Kompone...,"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...
9,32.5,Executive Assistant (f/m/d) Central Blades & V...,"Deutschland, Berlin",https://jobs.siemens-energy.com/de_DE/jobs/Fol...


## 8. Write to SQLite
Upserts results into `jobs` table. Uses `(source, source_job_id)` as the unique key — safe to re-run, won't create duplicates.

⚠️ **Only run this when you're happy with the scores above.**

In [ ]:
# Requires jobs table to exist — run "python setup_db.py" once from repo root (the top-level job_scraper/ folder) in terminal if not already done. should say "DB created"

In [20]:
def write_to_db(jobs, db_path, min_score):
    """
    Upserts jobs into SQLite.
    INSERT OR REPLACE updates existing rows if (source, source_job_id) already exists.
    Only writes jobs at or above min_score.
    """
    today = str(date.today())
    to_write = [j for j in jobs if j['score'] >= min_score]
    skipped = len(jobs) - len(to_write)

    conn = sqlite3.connect(db_path)
    try:
        conn.executemany("""
            INSERT OR REPLACE INTO jobs
                (run_date, source, source_job_id, company, title, location,
                 start_date, link, score, description)
            VALUES
                (:run_date, :source, :source_job_id, :company, :title, :location,
                 :start_date, :link, :score, :description)
        """, [{**j, 'run_date': today} for j in to_write])
        conn.commit()
        print(f'✅ Wrote {len(to_write)} jobs to {db_path}')
        print(f'   Skipped {skipped} jobs below MIN_SCORE ({min_score})')
    except Exception as e:
        conn.rollback()
        print(f'❌ DB write failed: {e}')
        raise
    finally:
        conn.close()


write_to_db(scored_jobs, DB_PATH, MIN_SCORE)

✅ Wrote 45 jobs to data/jobs.db
   Skipped 6 jobs below MIN_SCORE (15)


In [21]:
# Verify: query the DB to confirm rows were written
conn = sqlite3.connect(DB_PATH)
df_db = pd.read_sql(
    f"SELECT run_date, source, title, location, score FROM jobs WHERE source = '{SOURCE}' ORDER BY score DESC",
    conn
)
conn.close()
print(f'{len(df_db)} rows in DB for source={SOURCE}')
df_db.head(10)

45 rows in DB for source=siemens_energy


,run_date,source,title,location,score
0,2026-04-20,siemens_energy,Product Safety Officer (f/m/d),"Deutschland, Berlin",36.7
1,2026-04-20,siemens_energy,Pflichtpraktikant (w/m/d) - Leadership & Organ...,"Deutschland, Berlin",35.7
2,2026-04-20,siemens_energy,Head (f/m/d) of Applied Engineering & Testing,"Deutschland, Berlin",35.2
3,2026-04-20,siemens_energy,IPLM Lead (w/m/d) for Large GasTurbine,"Deutschland, Berlin",35.2
4,2026-04-20,siemens_energy,Global Order Manager (f/m/d),"Deutschland, Berlin",34.4
5,2026-04-20,siemens_energy,Commercial Project Manager (f/m/d) mit Digital...,"Deutschland, Berlin",33.3
6,2026-04-20,siemens_energy,Global Factory Network Costing Professional (m...,"Deutschland, Berlin",32.7
7,2026-04-20,siemens_energy,Logistikcore Mitarbeiter (w/m/d),"Deutschland, Berlin",32.6
8,2026-04-20,siemens_energy,Executive Assistant (f/m/d) Central Blades & V...,"Deutschland, Berlin",32.5
9,2026-04-20,siemens_energy,Projektleiter Verfahrenstechnik in der Kompone...,"Deutschland, Berlin",32.5
